# Param Shah
## 240905320
### CSE-D 33
#### Lab 9: Decision Tree [C4.5 and CART]

Q1) Write a python function program to demonstrate the working of the decision tree based  C4.5 algorithms 
without  using  scikit-learn  library.  Use  following  data  set  for  building  the  decision  tree  and  apply  this 
knowledge to classify a new sample.  
The dataset has three attributes: Outlook (Sunny, Overcast, Rainy), Temperature, Humidity and Wind (Weak, 
Strong). The target attribute is Play Tennis (Yes/No).

In [1]:
import pandas as pd
import math
from collections import Counter

df = pd.read_csv("tennis_conditions.csv")

attributes = ["Outlook", "Temperature", "Humidity", "Wind"]
numeric = {"Temperature", "Humidity"}
target = "PlayTennis"

def entropy(data):
    counts = Counter(data[target])
    total = len(data)

    return -sum(
        (count / total) * math.log2(count / total)
        for count in counts.values()
    )

def gain_ratio(data, attr):
    parent_entropy = entropy(data)

    if attr in numeric:
        values = sorted(data[attr].unique())

        best_ratio = -1
        best_threshold = None

        for i in range(len(values) - 1):
            threshold = (values[i] + values[i + 1]) / 2

            left = data[data[attr] <= threshold]
            right = data[data[attr] > threshold]

            if len(left) == 0 or len(right) == 0:
                continue

            subsets = [left, right]

            remainder = sum(
                len(s) / len(data) * entropy(s)
                for s in subsets
            )

            gain = parent_entropy - remainder

            split_info = -sum(
                (len(s) / len(data)) *
                math.log2(len(s) / len(data))
                for s in subsets
            )

            ratio = gain / split_info if split_info != 0 else 0

            if ratio > best_ratio:
                best_ratio = ratio
                best_threshold = threshold

        return best_ratio, best_threshold

    groups = [group for _, group in data.groupby(attr)]

    remainder = sum(
        len(group) / len(data) * entropy(group)
        for group in groups
    )

    gain = parent_entropy - remainder

    split_info = -sum(
        (len(group) / len(data)) *
        math.log2(len(group) / len(data))
        for group in groups
    )

    ratio = gain / split_info if split_info != 0 else 0

    return ratio, None

def c45(data, attrs):
    labels = data[target]

    if len(labels.unique()) == 1:
        return labels.iloc[0]

    if not attrs:
        return labels.mode()[0]

    best_attr = None
    best_ratio = -1
    best_threshold = None

    for attr in attrs:
        ratio, threshold = gain_ratio(data, attr)

        if ratio > best_ratio:
            best_ratio = ratio
            best_attr = attr
            best_threshold = threshold

    remaining = [a for a in attrs if a != best_attr]

    if best_attr in numeric:
        left = data[data[best_attr] <= best_threshold]
        right = data[data[best_attr] > best_threshold]

        return {
            best_attr: {
                "threshold": best_threshold,
                "<=": c45(left, remaining),
                ">": c45(right, remaining)
            }
        }

    tree = {best_attr: {}}

    for value in data[best_attr].unique():
        subset = data[data[best_attr] == value]
        tree[best_attr][value] = c45(subset, remaining)

    return tree

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attr = next(iter(tree))
    branches = tree[attr]

    if "threshold" in branches:
        threshold = branches["threshold"]

        if sample[attr] <= threshold:
            return predict(branches["<="], sample)
        else:
            return predict(branches[">"], sample)

    value = sample[attr]

    if value not in branches:
        return "Unknown"

    return predict(branches[value], sample)

tree = c45(df, attributes)

print("Decision Tree:")
print(tree)

new_sample = {
    "Outlook": "Sunny",
    "Temperature": 72,
    "Humidity": 90,
    "Wind": "Strong"
}

print("\nNew Sample:")
print(new_sample)

result = predict(tree, new_sample)

print("\nPrediction:", result)

Decision Tree:
{'Temperature': {'threshold': 84.0, '<=': {'Outlook': {'Sunny': {'Humidity': {'threshold': 80.0, '<=': 'Yes', '>': 'No'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}, '>': 'No'}}

New Sample:
{'Outlook': 'Sunny', 'Temperature': 72, 'Humidity': 90, 'Wind': 'Strong'}

Prediction: No


Q2) Write a python function program to demonstrate the working of the decision tree based  CART algorithms 
without using scikit-learn library. Use Q. No. 1  data set for building the decision tree and apply this knowledge 
to classify a new sample.  
The dataset has three attributes: Outlook (Sunny, Overcast, Rainy), Temperature, Humidity and Wind (Weak, 
Strong). The target attribute is Play Tennis (Yes/No).

In [2]:
import pandas as pd

df = pd.read_csv("tennis_conditions.csv")

attributes = ["Outlook", "Temperature", "Humidity", "Wind"]
numeric = {"Temperature", "Humidity"}
target = "PlayTennis"

def gini(data):
    if len(data) == 0:
        return 0

    probabilities = data[target].value_counts(normalize=True)
    return 1 - sum(p ** 2 for p in probabilities)

def best_split(data):
    best_attr = None
    best_value = None
    best_gini = float("inf")
    best_type = None

    for attr in attributes:
        if attr in numeric:
            values = sorted(data[attr].unique())

            for i in range(len(values) - 1):
                threshold = (values[i] + values[i + 1]) / 2

                left = data[data[attr] <= threshold]
                right = data[data[attr] > threshold]

                if len(left) == 0 or len(right) == 0:
                    continue

                score = (
                    len(left) / len(data) * gini(left)
                    + len(right) / len(data) * gini(right)
                )

                if score < best_gini:
                    best_gini = score
                    best_attr = attr
                    best_value = threshold
                    best_type = "numeric"

        else:
            for value in data[attr].unique():
                left = data[data[attr] == value]
                right = data[data[attr] != value]

                if len(left) == 0 or len(right) == 0:
                    continue

                score = (
                    len(left) / len(data) * gini(left)
                    + len(right) / len(data) * gini(right)
                )

                if score < best_gini:
                    best_gini = score
                    best_attr = attr
                    best_value = value
                    best_type = "categorical"

    return best_attr, best_value, best_type

def cart(data, depth=0, max_depth=5):
    labels = data[target]

    if len(labels.unique()) == 1:
        return labels.iloc[0]

    if depth >= max_depth:
        return labels.mode()[0]

    attr, value, split_type = best_split(data)

    if attr is None:
        return labels.mode()[0]

    if split_type == "numeric":
        left = data[data[attr] <= value]
        right = data[data[attr] > value]

        return {
            "attribute": attr,
            "type": "numeric",
            "value": value,
            "left": cart(left, depth + 1, max_depth),
            "right": cart(right, depth + 1, max_depth)
        }

    left = data[data[attr] == value]
    right = data[data[attr] != value]

    return {
        "attribute": attr,
        "type": "categorical",
        "value": value,
        "left": cart(left, depth + 1, max_depth),
        "right": cart(right, depth + 1, max_depth)
    }

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attr = tree["attribute"]
    value = tree["value"]

    if tree["type"] == "numeric":
        if sample[attr] <= value:
            return predict(tree["left"], sample)
        return predict(tree["right"], sample)

    if sample[attr] == value:
        return predict(tree["left"], sample)
    return predict(tree["right"], sample)

tree = cart(df)

print("CART Decision Tree:")
print(tree)

new_sample = {
    "Outlook": "Sunny",
    "Temperature": 72,
    "Humidity": 90,
    "Wind": "Strong"
}

print("\nNew Sample:")
print(new_sample)

result = predict(tree, new_sample)

print("\nPrediction:", result)

CART Decision Tree:
{'attribute': 'Outlook', 'type': 'categorical', 'value': 'Overcast', 'left': 'Yes', 'right': {'attribute': 'Temperature', 'type': 'numeric', 'value': 77.5, 'left': {'attribute': 'Temperature', 'type': 'numeric', 'value': 66.5, 'left': 'No', 'right': {'attribute': 'Temperature', 'type': 'numeric', 'value': 70.5, 'left': 'Yes', 'right': {'attribute': 'Temperature', 'type': 'numeric', 'value': 73.5, 'left': 'No', 'right': 'Yes'}}}, 'right': 'No'}}

New Sample:
{'Outlook': 'Sunny', 'Temperature': 72, 'Humidity': 90, 'Wind': 'Strong'}

Prediction: No


Q3) Write a python function program to demonstrate the working of the decision tree based  C4.5 and CART 
algorithms  without  and  with  using  scikit-learn  library.  Using  the  following  dataset,  apply  aforementioned 
algorithms. The attributes are Income (Low, Medium, High) and Credit (Good, Bad), and the target is Loan 
Approved (Yes/No).

In [3]:
import pandas as pd
import math
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("loan.csv")

attributes = ["Income", "Credit"]
target = "LoanApproved"

def entropy(data):
    counts = Counter(data[target])
    total = len(data)

    return -sum(
        (c / total) * math.log2(c / total)
        for c in counts.values()
    )

def gain_ratio(data, attr):
    parent = entropy(data)

    groups = [group for _, group in data.groupby(attr)]

    remainder = sum(
        len(group) / len(data) * entropy(group)
        for group in groups
    )

    gain = parent - remainder

    split_info = -sum(
        (len(group) / len(data)) *
        math.log2(len(group) / len(data))
        for group in groups
    )

    return gain / split_info if split_info else 0

def c45(data, attrs):
    labels = data[target]

    if len(labels.unique()) == 1:
        return labels.iloc[0]

    if not attrs:
        return labels.mode()[0]

    best = max(attrs, key=lambda a: gain_ratio(data, a))

    tree = {best: {}}

    remaining = [a for a in attrs if a != best]

    for value in data[best].unique():
        subset = data[data[best] == value]
        tree[best][value] = c45(subset, remaining)

    return tree

def gini(data):
    probabilities = data[target].value_counts(normalize=True)
    return 1 - sum(p ** 2 for p in probabilities)

def best_cart_split(data):
    best_attr = None
    best_value = None
    best_score = float("inf")

    for attr in attributes:
        for value in data[attr].unique():

            left = data[data[attr] == value]
            right = data[data[attr] != value]

            if len(left) == 0 or len(right) == 0:
                continue

            score = (
                len(left) / len(data) * gini(left)
                + len(right) / len(data) * gini(right)
            )

            if score < best_score:
                best_score = score
                best_attr = attr
                best_value = value

    return best_attr, best_value

def cart(data):
    labels = data[target]

    if len(labels.unique()) == 1:
        return labels.iloc[0]

    attr, value = best_cart_split(data)

    if attr is None:
        return labels.mode()[0]

    left = data[data[attr] == value]
    right = data[data[attr] != value]

    return {
        attr: {
            value: cart(left),
            "Others": cart(right)
        }
    }

print("C4.5 without sklearn:")
print(c45(df, attributes))

print("\nCART without sklearn:")
print(cart(df))

encoded = df.copy()

encoders = {}

for column in ["Income", "Credit", "LoanApproved"]:
    le = LabelEncoder()
    encoded[column] = le.fit_transform(encoded[column])
    encoders[column] = le

X = encoded[["Income", "Credit"]]
y = encoded["LoanApproved"]

c45_model = DecisionTreeClassifier(criterion="entropy", random_state=0)
c45_model.fit(X, y)

cart_model = DecisionTreeClassifier(criterion="gini", random_state=0)
cart_model.fit(X, y)

sample = pd.DataFrame({
    "Income": [
        encoders["Income"].transform(["Medium"])[0]
    ],
    "Credit": [
        encoders["Credit"].transform(["Bad"])[0]
    ]
})

c45_prediction = c45_model.predict(sample)[0]
cart_prediction = cart_model.predict(sample)[0]

print("\nNew Sample:")
print("Income = Medium, Credit = Bad")

print(
    "\nC4.5 sklearn prediction:",
    encoders["LoanApproved"].inverse_transform([c45_prediction])[0]
)

print(
    "CART sklearn prediction:",
    encoders["LoanApproved"].inverse_transform([cart_prediction])[0]
)

C4.5 without sklearn:
{'Credit': {'Good': 'Yes', 'Bad': {'Income': {'Low': 'No', 'Medium': 'Yes', 'High': 'No'}}}}

CART without sklearn:
{'Credit': {'Good': 'Yes', 'Others': {'Income': {'Medium': 'Yes', 'Others': 'No'}}}}

New Sample:
Income = Medium, Credit = Bad

C4.5 sklearn prediction: Yes
CART sklearn prediction: Yes
